In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd() / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'data'

In [2]:
full_merged_data = pd.read_csv(DATA_DIR / 'merged_geo.csv')

In [5]:
# Pull additional public high school attributes from CCD directory (2019-2023)
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Use a Session + retries/backoff to handle occasional dropped/partial responses
session = requests.Session()
retries = Retry(
    total=8,
    connect=8,
    read=8,
    status=8,
    backoff_factor=0.75,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)
session.mount("https://", HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10))
session.headers.update({"User-Agent": "c2i-ccd-directory/1.0"})

def fetch_json(url: str) -> dict:
    for attempt in range(6):
        try:
            resp = session.get(url, timeout=(15, 120))
            resp.raise_for_status()
            return resp.json()
        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError):
            if attempt == 5:
                raise
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError("Unreachable")

years = [2019, 2020, 2021, 2022, 2023]
year_dfs = []

for year in years:
    url = f"https://educationdata.urban.org/api/v1/schools/ccd/directory/{year}/"
    rows = []
    while url:
        data = fetch_json(url)
        rows.extend(data.get('results', []))
        url = data.get('next')
        time.sleep(0.05)

    df_year = pd.DataFrame(rows)
    df_year['year'] = year
    year_dfs.append(df_year)
    print(f"Finished CCD directory year {year} (rows: {len(rows):,})")

ccd_dir = pd.concat(year_dfs, ignore_index=True)
ccd_dir.columns = [c.lower() for c in ccd_dir.columns]

needed = [
    'free_or_reduced_price_lunch',
    'title_i_status',
    'urban_centric_locale',
    'charter',
    'magnet',
    'teachers_fte',
    'enrollment',
]

# Keep only columns that exist (API fields can vary slightly by year)
keep_cols = [c for c in ['year', 'ncessch'] + needed if c in ccd_dir.columns]
ccd_dir = ccd_dir[keep_cols].copy()

# Standardize keys to align with your main dataset later
ccd_dir['hs_id'] = ccd_dir['ncessch'].astype('string').str.strip()
ccd_dir['cycle'] = pd.to_numeric(ccd_dir['year'], errors='coerce').astype('Int64')

rename_map = {c: f"hs_{c}" for c in needed if c in ccd_dir.columns}
ccd_hs_econ = ccd_dir.rename(columns=rename_map)
ccd_hs_econ = ccd_hs_econ.drop(columns=[c for c in ['ncessch', 'year'] if c in ccd_hs_econ.columns])

print('ccd_hs_econ shape:', ccd_hs_econ.shape)
ccd_hs_econ.head()

Finished CCD directory year 2019 (rows: 101,688)
Finished CCD directory year 2020 (rows: 101,662)
Finished CCD directory year 2021 (rows: 102,130)
Finished CCD directory year 2022 (rows: 102,268)
Finished CCD directory year 2023 (rows: 102,274)
ccd_hs_econ shape: (510022, 9)


,hs_free_or_reduced_price_lunch,hs_title_i_status,hs_urban_centric_locale,hs_charter,hs_magnet,hs_teachers_fte,hs_enrollment,hs_id,cycle
0,608.0,3.0,32,0,0.0,42.0,861.0,010000500870,2019
1,901.0,3.0,32,0,0.0,80.0,1554.0,010000500871,2019
2,649.0,3.0,32,0,0.0,41.0,908.0,010000500879,2019
3,706.0,3.0,32,0,0.0,50.0,934.0,010000500889,2019
4,384.0,3.0,32,0,0.0,29.0,607.0,010000501616,2019


In [6]:
# Derived metrics
for c in ['hs_free_or_reduced_price_lunch', 'hs_enrollment', 'hs_teachers_fte']:
    if c in ccd_hs_econ.columns:
        ccd_hs_econ[c] = pd.to_numeric(ccd_hs_econ[c], errors='coerce')

if {'hs_free_or_reduced_price_lunch', 'hs_enrollment'}.issubset(ccd_hs_econ.columns):
    ccd_hs_econ['hs_pct_free_or_reduced_price_lunch'] = ccd_hs_econ['hs_free_or_reduced_price_lunch'] / ccd_hs_econ['hs_enrollment']
    ccd_hs_econ.loc[ccd_hs_econ['hs_enrollment'].isna() | (ccd_hs_econ['hs_enrollment'] == 0), 'hs_pct_free_or_reduced_price_lunch'] = pd.NA

if {'hs_enrollment', 'hs_teachers_fte'}.issubset(ccd_hs_econ.columns):
    ccd_hs_econ['hs_students_per_teacher'] = ccd_hs_econ['hs_enrollment'] / ccd_hs_econ['hs_teachers_fte']
    ccd_hs_econ.loc[ccd_hs_econ['hs_teachers_fte'].isna() | (ccd_hs_econ['hs_teachers_fte'] == 0), 'hs_students_per_teacher'] = pd.NA

ccd_hs_econ.head()

,hs_free_or_reduced_price_lunch,hs_title_i_status,hs_urban_centric_locale,hs_charter,hs_magnet,hs_teachers_fte,hs_enrollment,hs_id,cycle,hs_pct_free_or_reduced_price_lunch,hs_students_per_teacher
0,608.0,3.0,32,0,0.0,42.0,861.0,010000500870,2019,0.706156,20.500000
1,901.0,3.0,32,0,0.0,80.0,1554.0,010000500871,2019,0.579794,19.425000
2,649.0,3.0,32,0,0.0,41.0,908.0,010000500879,2019,0.714758,22.146341
3,706.0,3.0,32,0,0.0,50.0,934.0,010000500889,2019,0.755889,18.680000
4,384.0,3.0,32,0,0.0,29.0,607.0,010000501616,2019,0.632619,20.931034


In [7]:
ccd_hs_econ.dtypes

hs_free_or_reduced_price_lunch        float64
hs_title_i_status                      object
hs_urban_centric_locale                 int64
hs_charter                              int64
hs_magnet                              object
hs_teachers_fte                       float64
hs_enrollment                         float64
hs_id                                  string
cycle                                   Int64
hs_pct_free_or_reduced_price_lunch    float64
hs_students_per_teacher               float64
dtype: object

In [8]:
# Cast selected columns and drop raw inputs
for col in ['hs_title_i_status', 'hs_magnet']:
    if col in ccd_hs_econ.columns:
        ccd_hs_econ[col] = pd.to_numeric(ccd_hs_econ[col], errors='coerce')
        # Use int64 when possible; otherwise keep nullable Int64
        if ccd_hs_econ[col].isna().any():
            ccd_hs_econ[col] = ccd_hs_econ[col].astype('Int64')
        else:
            ccd_hs_econ[col] = ccd_hs_econ[col].astype('int64')

ccd_hs_econ.drop(columns=['hs_free_or_reduced_price_lunch', 'hs_teachers_fte'], inplace=True, errors='ignore')
ccd_hs_econ.dtypes

hs_title_i_status                       Int64
hs_urban_centric_locale                 int64
hs_charter                              int64
hs_magnet                               Int64
hs_enrollment                         float64
hs_id                                  string
cycle                                   Int64
hs_pct_free_or_reduced_price_lunch    float64
hs_students_per_teacher               float64
dtype: object

In [9]:
# Merge CCD HS economic attributes into the full merged dataset on hs_id + cycle
full_merged_data['hs_id'] = full_merged_data['hs_id'].astype('string').str.strip()
full_merged_data['cycle'] = pd.to_numeric(full_merged_data['cycle'], errors='coerce').astype('Int64')

ccd_hs_econ['hs_id'] = ccd_hs_econ['hs_id'].astype('string').str.strip()
ccd_hs_econ['cycle'] = pd.to_numeric(ccd_hs_econ['cycle'], errors='coerce').astype('Int64')

# Ensure uniqueness on join keys to avoid row explosions
ccd_hs_econ_dedup = ccd_hs_econ.drop_duplicates(subset=['hs_id', 'cycle'])

full_merged_with_hs_econ = full_merged_data.merge(
    ccd_hs_econ_dedup,
    on=['hs_id', 'cycle'],
    how='left'
)

print('full_merged_data shape:', full_merged_data.shape)
print('full_merged_with_hs_econ shape:', full_merged_with_hs_econ.shape)
full_merged_with_hs_econ.head()

full_merged_data shape: (1048575, 31)
full_merged_with_hs_econ shape: (1048575, 38)


,college_id,cycle,school_type,hs_id,col_name,col_city,col_st,col_zip,col_type,col_ctyname,...,hs_pct_aian,hs_pct_nhpi,hs_pct_two_or_more,hs_title_i_status,hs_urban_centric_locale,hs_charter,hs_magnet,hs_enrollment,hs_pct_free_or_reduced_price_lunch,hs_students_per_teacher
0,45896401,2022,public,450201000242,STRAYER UNIVERSITY - CHARLESTON CAMPUS,NORTH CHARLESTON,SC,29418.0,3.0,CHARLESTON,...,0.000000,0.001176,0.088235,<NA>,21.0,0.0,<NA>,850.0,0.757647,14.655172
1,101189,2023,private,A1900014,FAULKNER UNIVERSITY,MONTGOMERY,AL,36109.0,2.0,MONTGOMERY,...,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
2,46114802,2020,public,360007705522,NEW YORK FILM ACADEMY - NEW YORK CAMPUS,NEW YORK,NY,10004.0,3.0,NEW YORK,...,0.000000,0.000000,0.000000,5,11.0,0.0,0,58.0,0.896552,3.866667
3,200800,2022,public,390434800040,UNIVERSITY OF AKRON MAIN CAMPUS,AKRON,OH,44325.0,1.0,SUMMIT,...,0.000000,0.005391,0.086253,<NA>,12.0,0.0,<NA>,371.0,NaN,12.793103
4,155089,2020,public,201113000926,FRIENDS UNIVERSITY,WICHITA,KS,67213.0,2.0,SEDGWICK,...,0.110599,0.004608,0.073733,4,31.0,0.0,0,217.0,0.460829,14.466667


In [11]:
full_merged_with_hs_econ.to_csv(DATA_DIR / 'merge_hs_econ.csv', index=False)